# Kaggle Titanic Dataset (HistGradientBoosting)

In [15]:
# 1) Imports and Paths
import os
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
#from sklearn.experimental import enable_hist_gradient_boosting  # noqa: F401
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_val_predict, RandomizedSearchCV
from sklearn.calibration import CalibratedClassifierCV
from sklearn.inspection import permutation_importance
from sklearn import metrics
import joblib

# Paths and constants
DATA_DIR = "/home/atul-kumar/workspace/kaggle/titanic/data"
TRAIN_PATH = os.path.join(DATA_DIR, "train.csv")
TEST_PATH = os.path.join(DATA_DIR, "test.csv")
SUBMISSION_PATH = os.path.join(DATA_DIR, "submission-hgb.csv")
MODEL_DIR = os.path.join(DATA_DIR, "models")
os.makedirs(MODEL_DIR, exist_ok=True)

RANDOM_STATE = 63
np.random.seed(RANDOM_STATE)
print("Paths set:", TRAIN_PATH, TEST_PATH)

Paths set: /home/atul-kumar/workspace/kaggle/titanic/data/train.csv /home/atul-kumar/workspace/kaggle/titanic/data/test.csv


In [16]:
# 2) Load Data
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
print("Train shape:", train_df.shape, " Test shape:", test_df.shape)
train_df.head(3)

Train shape: (891, 12)  Test shape: (418, 11)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S


In [17]:
# 3) Enhanced Feature Engineering (reuse from RF notebook)

def add_engineered_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    # Title + bucket
    out["Title"] = out["Name"].str.extract(r",\s*([^\.]+)\.")
    title_map = {
        'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs',
        'Lady': 'Rare', 'Countess': 'Rare', 'Dona': 'Rare', 'Sir': 'Rare', 'Don': 'Rare',
        'Jonkheer': 'Rare', 'Capt': 'Rare', 'Col': 'Rare', 'Dr': 'Rare', 'Rev': 'Rare',
        'Major': 'Rare'
    }
    out["TitleBucket"] = out["Title"].replace(title_map)
    out.loc[~out["TitleBucket"].isin(['Mr', 'Mrs', 'Miss', 'Master', 'Rare']), "TitleBucket"] = 'Rare'

    # Family
    out["FamilySize"] = out.get("SibSp", 0) + out.get("Parch", 0) + 1
    out["IsAlone"] = (out["FamilySize"] == 1).astype(int)
    def _family_bin(n):
        if n == 1: return 'Single'
        if 2 <= n <= 4: return 'Small'
        return 'Large'
    out["FamilySizeBin"] = out["FamilySize"].apply(_family_bin)

    # Ticket
    if "Ticket" in out.columns:
        counts = out["Ticket"].value_counts()
        out["TicketGroup"] = out["Ticket"].map(counts)
        prefix = out["Ticket"].astype(str).str.replace(r"[^A-Za-z]+", "", regex=True).str.upper()
        prefix = prefix.replace("", np.nan).fillna("NONE")
        pref_counts = prefix.value_counts()
        common = set(pref_counts[pref_counts >= 10].index)
        prefix = prefix.where(prefix.isin(common), other="RARE")
        out["TicketPrefix"] = prefix
    else:
        out["TicketGroup"] = 1
        out["TicketPrefix"] = "NONE"

    # Cabin
    cabin = out.get("Cabin")
    out["CabinKnown"] = cabin.notna().astype(int)
    out["CabinDeck"] = cabin.astype(str).str[0]
    out["CabinDeck"] = out["CabinDeck"].where(out["CabinKnown"] == 1, other='U')

    # Fare/age transforms
    out["FareLog"] = np.log1p(out["Fare"]) if "Fare" in out.columns else 0.0
    out["AgePclass"] = out.get("Age", np.nan) * out.get("Pclass", np.nan)
    age_bins = [-1, 12, 18, 35, 60, 100]
    age_labels = ["child", "teen", "young", "adult", "senior"]
    out["AgeBand"] = pd.cut(out["Age"], bins=age_bins, labels=age_labels)

    return out

train_df_fe = add_engineered_features(train_df)
test_df_fe = add_engineered_features(test_df)
print("Engineered columns present:", {c for c in train_df_fe.columns if c in [
    "Title","TitleBucket","FamilySize","IsAlone","FamilySizeBin","TicketGroup",
    "TicketPrefix","CabinKnown","CabinDeck","FareLog","AgePclass","AgeBand"
]})

Engineered columns present: {'TicketGroup', 'CabinKnown', 'FamilySizeBin', 'CabinDeck', 'AgeBand', 'FareLog', 'IsAlone', 'TitleBucket', 'FamilySize', 'Title', 'AgePclass', 'TicketPrefix'}


In [18]:
# 4) Define Preprocessing
base_features = ["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked"]
engineered = [
    "TitleBucket", "FamilySize", "IsAlone", "FamilySizeBin", "TicketGroup", "TicketPrefix",
    "CabinKnown", "CabinDeck", "FareLog", "AgePclass", "AgeBand"
]
all_features = base_features + engineered

numeric_features = ["Age", "SibSp", "Parch", "Fare", "FamilySize", "TicketGroup", "FareLog", "AgePclass"]
categorical_features = ["Pclass", "Sex", "Embarked", "TitleBucket", "FamilySizeBin", 
                        "TicketPrefix", "CabinKnown", "CabinDeck", "AgeBand"]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

X = train_df_fe[all_features]
y = train_df_fe["Survived"]
print("Feature matrix shape:", X.shape)

Feature matrix shape: (891, 18)


In [19]:
# 5) HistGradientBoosting Model Definition
hgb = HistGradientBoostingClassifier(
    learning_rate=0.1,
    max_depth=6,
    max_leaf_nodes=31,
    min_samples_leaf=20,
    l2_regularization=0.0,
    max_bins=255,
    random_state=RANDOM_STATE
)

model = Pipeline(steps=[
    ("pre", preprocess),
    ("clf", hgb),
])
print(model)

Pipeline(steps=[('pre',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['Age', 'SibSp', 'Parch',
                                                   'Fare', 'FamilySize',
                                                   'TicketGroup', 'FareLog',
                                                   'AgePclass']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore',
                  

In [20]:
# 6) Cross-Validation Metrics (Accuracy and ROC-AUC)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
acc_scores = cross_val_score(model, X, y, cv=cv, scoring='accuracy', n_jobs=-1)
auc_scores = cross_val_score(model, X, y, cv=cv, scoring='roc_auc', n_jobs=-1)
print(f"CV Accuracy: {acc_scores.mean():.4f} +/- {acc_scores.std():.4f}")
print(f"CV ROC-AUC: {auc_scores.mean():.4f} +/- {auc_scores.std():.4f}")
print("AUC is the area under ROC curve: ∫_0^1 TPR(FPR^{-1}(x)) dx")

CV Accuracy: 0.8271 +/- 0.0111
CV ROC-AUC: 0.8588 +/- 0.0198
AUC is the area under ROC curve: ∫_0^1 TPR(FPR^{-1}(x)) dx


In [21]:
# 7) Hyperparameter Search with RandomizedSearchCV
param_distributions = {
    'clf__learning_rate': [0.03, 0.05, 0.08, 0.1, 0.15, 0.2],
    'clf__max_depth': [3, 4, 5, 6, 8],
    'clf__max_leaf_nodes': [15, 31, 63],
    'clf__min_samples_leaf': [10, 20, 30, 50, 80],
    'clf__l2_regularization': [0.0, 0.01, 0.1, 0.5, 1.0],
    'clf__max_bins': [128, 255],
}

search = RandomizedSearchCV(
    estimator=model,
    param_distributions=param_distributions,
    n_iter=40,
    scoring='roc_auc',
    cv=cv,
    n_jobs=-1,
    random_state=RANDOM_STATE,
    refit=True,
    verbose=1,
)

search.fit(X, y)
print("Best params:", search.best_params_)
print("Best ROC-AUC:", round(search.best_score_, 4))

Fitting 5 folds for each of 40 candidates, totalling 200 fits
Best params: {'clf__min_samples_leaf': 50, 'clf__max_leaf_nodes': 15, 'clf__max_depth': 8, 'clf__max_bins': 128, 'clf__learning_rate': 0.08, 'clf__l2_regularization': 0.1}
Best ROC-AUC: 0.8759
Best params: {'clf__min_samples_leaf': 50, 'clf__max_leaf_nodes': 15, 'clf__max_depth': 8, 'clf__max_bins': 128, 'clf__learning_rate': 0.08, 'clf__l2_regularization': 0.1}
Best ROC-AUC: 0.8759


In [22]:
# 8) Refit Best Model on Full Training Data
best_estimator = search.best_estimator_
best_estimator.fit(X, y)
final_model = best_estimator
print("Best HGB model refit on full data.")

Best HGB model refit on full data.


In [23]:
# 9) Probability Calibration (Optional)
calibrated_model = CalibratedClassifierCV(estimator=final_model, method='sigmoid', cv=5)
logloss_cv_final = -cross_val_score(final_model, X, y, cv=cv, scoring='neg_log_loss', n_jobs=-1)
logloss_cv_calibrated = -cross_val_score(calibrated_model, X, y, cv=cv, scoring='neg_log_loss', n_jobs=-1)
print(f"LogLoss final: {logloss_cv_final.mean():.4f} +/- {logloss_cv_final.std():.4f}")
print(f"LogLoss calibrated: {logloss_cv_calibrated.mean():.4f} +/- {logloss_cv_calibrated.std():.4f}")

chosen_model = calibrated_model if logloss_cv_calibrated.mean() < logloss_cv_final.mean() else final_model
print("Chosen model:", "calibrated" if chosen_model is calibrated_model else "final (uncalibrated)")

LogLoss final: 0.4225 +/- 0.0370
LogLoss calibrated: 0.4185 +/- 0.0230
Chosen model: calibrated


In [24]:
# 10) Threshold Tuning on Validation Predictions
p_oof = cross_val_predict(chosen_model, X, y, cv=cv, method='predict_proba', n_jobs=-1)[:, 1]

best_threshold = 0.5
best_f1 = -1
for t in np.linspace(0.2, 0.8, 121):
    preds = (p_oof >= t).astype(int)
    f1 = metrics.f1_score(y, preds)
    if f1 > best_f1:
        best_f1 = f1
        best_threshold = float(t)

acc = metrics.accuracy_score(y, (p_oof >= best_threshold).astype(int))
print(f"Best threshold: {best_threshold:.3f} with F1={best_f1:.4f}, Accuracy={acc:.4f}")

Best threshold: 0.385 with F1=0.7787, Accuracy=0.8272


In [25]:
# 11) Permutation Feature Importance (input level)
from sklearn.model_selection import train_test_split
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)
chosen_model.fit(X_train, y_train)
r = permutation_importance(chosen_model, X_valid, y_valid, scoring='roc_auc', n_repeats=10, random_state=RANDOM_STATE, n_jobs=-1)
feature_names = list(all_features)
importances = pd.Series(r.importances_mean, index=feature_names).sort_values(ascending=False)
print("Top 15 features by permutation importance (input level):")
importances.head(15)

Top 15 features by permutation importance (input level):


TitleBucket      0.038478
Sex              0.037273
Pclass           0.026943
Fare             0.024506
AgePclass        0.010171
CabinKnown       0.008518
Embarked         0.005922
FamilySize       0.003215
TicketPrefix     0.003004
FamilySizeBin    0.000935
Parch            0.000837
TicketGroup      0.000540
SibSp            0.000138
IsAlone          0.000000
FareLog          0.000000
dtype: float64

In [26]:
# 12) Predict Test Set and Save Submission
X_test = test_df_fe[all_features]
probs_test = chosen_model.predict_proba(X_test)[:, 1]
labels_test = (probs_test >= best_threshold).astype(int)

submission = pd.DataFrame({
    'PassengerId': test_df_fe['PassengerId'],
    'Survived': labels_test
})
submission.to_csv(SUBMISSION_PATH, index=False)
print("Saved submission to:", SUBMISSION_PATH)
submission.head(10)

Saved submission to: /home/atul-kumar/workspace/kaggle/titanic/data/submission-hgb.csv


,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,1
5,897,0
6,898,1
7,899,0
8,900,1
9,901,0


In [27]:
# 13) Save Trained Model Artifact
metadata = {
    'best_params': getattr(search, 'best_params_', None),
    'best_threshold': best_threshold,
    'cv_accuracy': float(np.mean(acc_scores)),
    'cv_auc': float(np.mean(auc_scores)),
}
artifact_path = os.path.join(MODEL_DIR, 'hist-gradient-boosting-titanic.joblib')
joblib.dump({'model': chosen_model, 'metadata': metadata}, artifact_path)
print("Saved model artifact to:", artifact_path)
metadata

Saved model artifact to: /home/atul-kumar/workspace/kaggle/titanic/data/models/hist-gradient-boosting-titanic.joblib


{'best_params': {'clf__min_samples_leaf': 50,
  'clf__max_leaf_nodes': 15,
  'clf__max_depth': 8,
  'clf__max_bins': 128,
  'clf__learning_rate': 0.08,
  'clf__l2_regularization': 0.1},
 'best_threshold': 0.385,
 'cv_accuracy': 0.8271483271608814,
 'cv_auc': 0.8588366726463585}

In [28]:
# 14) Reproducibility: Random Seeds and Determinism
os.environ['PYTHONHASHSEED'] = str(RANDOM_STATE)
os.environ.setdefault('OMP_NUM_THREADS', '1')
print("Seeds and environment set for reproducibility.")

Seeds and environment set for reproducibility.
